In [2]:
from pathlib import Path
import ast
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wfdb

from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)


In [3]:


DATA_DIR = Path.cwd().parent/ "data"

METADATA_FILE = DATA_DIR / "ptbxl_database.csv"
STATEMENTS_FILE = DATA_DIR / "scp_statements.csv"

print("Dataset directory:", DATA_DIR)
print("Metadata exists:", METADATA_FILE.exists())
print("Diagnostic statements exist:", STATEMENTS_FILE.exists())

Dataset directory: C:\Users\pmondal\Documents\CardioTrustAI\src\CardioTrustAI.Backend\CardioTrustAI.PythonAPI\data
Metadata exists: True
Diagnostic statements exist: True


In [4]:

df = pd.read_csv(METADATA_FILE)
scp = pd.read_csv(STATEMENTS_FILE, index_col=0)

In [5]:
print("Total ECG records:", len(df))
print("Unique patients:", df["patient_id"].nunique())

Total ECG records: 21799
Unique patients: 18869


In [6]:
display(df.head(10))

,ecg_id,patient_id,age,sex,height,weight,nurse,site,device,recording_date,...,validated_by_human,baseline_drift,static_noise,burst_noise,electrodes_problems,extra_beats,pacemaker,strat_fold,filename_lr,filename_hr
0,1,15709.0,56.0,1,NaN,63.0,2.0,0.0,CS-12 E,1984-11-09 09:17:34,...,True,NaN,", I-V1,",NaN,NaN,NaN,NaN,3,records100/00000/00001_lr,records500/00000/00001_hr
1,2,13243.0,19.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-11-14 12:55:37,...,True,NaN,NaN,NaN,NaN,NaN,NaN,2,records100/00000/00002_lr,records500/00000/00002_hr
2,3,20372.0,37.0,1,NaN,69.0,2.0,0.0,CS-12 E,1984-11-15 12:49:10,...,True,NaN,NaN,NaN,NaN,NaN,NaN,5,records100/00000/00003_lr,records500/00000/00003_hr
3,4,17014.0,24.0,0,NaN,82.0,2.0,0.0,CS-12 E,1984-11-15 13:44:57,...,True,", II,III,AVF",NaN,NaN,NaN,NaN,NaN,3,records100/00000/00004_lr,records500/00000/00004_hr
4,5,17448.0,19.0,1,NaN,70.0,2.0,0.0,CS-12 E,1984-11-17 10:43:15,...,True,", III,AVR,AVF",NaN,NaN,NaN,NaN,NaN,4,records100/00000/00005_lr,records500/00000/00005_hr
5,6,19005.0,18.0,1,NaN,58.0,2.0,0.0,CS-12 E,1984-11-28 13:32:13,...,True,", V1",NaN,NaN,NaN,NaN,NaN,4,records100/00000/00006_lr,records500/00000/00006_hr
6,7,16193.0,54.0,0,NaN,83.0,2.0,0.0,CS-12 E,1984-11-28 13:32:22,...,True,NaN,NaN,NaN,NaN,NaN,NaN,7,records100/00000/00007_lr,records500/00000/00007_hr
7,8,11275.0,48.0,0,NaN,95.0,2.0,0.0,CS-12 E,1984-12-01 14:49:52,...,True,", II,AVF",", I-AVF,",NaN,NaN,NaN,NaN,9,records100/00000/00008_lr,records500/00000/00008_hr
8,9,18792.0,55.0,0,NaN,70.0,2.0,0.0,CS-12 E,1984-12-08 09:44:43,...,True,NaN,", I-AVR,",NaN,NaN,NaN,NaN,10,records100/00000/00009_lr,records500/00000/00009_hr
9,10,9456.0,22.0,1,NaN,56.0,2.0,0.0,CS-12 E,1984-12-12 14:12:46,...,True,NaN,NaN,NaN,NaN,NaN,NaN,9,records100/00000/00010_lr,records500/00000/00010_hr


In [7]:
print(df.columns.tolist())

['ecg_id', 'patient_id', 'age', 'sex', 'height', 'weight', 'nurse', 'site', 'device', 'recording_date', 'report', 'scp_codes', 'heart_axis', 'infarction_stadium1', 'infarction_stadium2', 'validated_by', 'second_opinion', 'initial_autogenerated_report', 'validated_by_human', 'baseline_drift', 'static_noise', 'burst_noise', 'electrodes_problems', 'extra_beats', 'pacemaker', 'strat_fold', 'filename_lr', 'filename_hr']


In [8]:
print(scp["diagnostic_class"].value_counts(dropna=False))

diagnostic_class
NaN     27
MI      14
STTC    13
CD      11
HYP      5
NORM     1
Name: count, dtype: int64


In [9]:
print(scp[["description", "diagnostic", "diagnostic_class"]].head(20))

                                               description  diagnostic  \
NDT                         non-diagnostic T abnormalities         1.0   
NST_                               non-specific ST changes         1.0   
DIG                                       digitalis-effect         1.0   
LNGQT                                     long QT-interval         1.0   
NORM                                            normal ECG         1.0   
IMI                         inferior myocardial infarction         1.0   
ASMI                    anteroseptal myocardial infarction         1.0   
LVH                           left ventricular hypertrophy         1.0   
LAFB                        left anterior fascicular block         1.0   
ISC_                                 non-specific ischemic         1.0   
IRBBB                 incomplete right bundle branch block         1.0   
1AVB                                 first degree AV block         1.0   
IVCD     non-specific intraventricular

In [10]:
diagnostic_classes = scp[
    scp["diagnostic"] == 1
]["diagnostic_class"].value_counts(dropna=False)

In [11]:
print(diagnostic_classes)

diagnostic_class
MI      14
STTC    13
CD      11
HYP      5
NORM     1
Name: count, dtype: int64


In [12]:
print(df[["ecg_id", "patient_id", "age", "sex", "scp_codes"]].head(10))

   ecg_id  patient_id   age  sex                                 scp_codes
0       1     15709.0  56.0    1  {'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}
1       2     13243.0  19.0    0              {'NORM': 80.0, 'SBRAD': 0.0}
2       3     20372.0  37.0    1                {'NORM': 100.0, 'SR': 0.0}
3       4     17014.0  24.0    0                {'NORM': 100.0, 'SR': 0.0}
4       5     17448.0  19.0    1                {'NORM': 100.0, 'SR': 0.0}
5       6     19005.0  18.0    1                {'NORM': 100.0, 'SR': 0.0}
6       7     16193.0  54.0    0                {'NORM': 100.0, 'SR': 0.0}
7       8     11275.0  48.0    0    {'IMI': 35.0, 'ABQRS': 0.0, 'SR': 0.0}
8       9     18792.0  55.0    0                {'NORM': 100.0, 'SR': 0.0}
9      10      9456.0  22.0    1                {'NORM': 100.0, 'SR': 0.0}


In [15]:


# Convert scp_codes from text to Python dictionaries
df["scp_codes"] = df["scp_codes"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

# Get only diagnostic statements
diagnostic_scp = scp[scp["diagnostic"] == 1]

# Map each ECG's diagnostic codes to their diagnostic classes
def get_diagnostic_classes(codes):
    classes = set()

    for code in codes:
        if code in diagnostic_scp.index:
            diagnostic_class = diagnostic_scp.loc[code, "diagnostic_class"]

            if isinstance(diagnostic_class, str):
                classes.add(diagnostic_class)

    return sorted(classes)

df["diagnostic_classes"] = df["scp_codes"].apply(
    get_diagnostic_classes
)

print(
    df[["ecg_id", "scp_codes", "diagnostic_classes"]].head(10)
)

   ecg_id                                 scp_codes diagnostic_classes
0       1  {'NORM': 100.0, 'LVOLT': 0.0, 'SR': 0.0}             [NORM]
1       2              {'NORM': 80.0, 'SBRAD': 0.0}             [NORM]
2       3                {'NORM': 100.0, 'SR': 0.0}             [NORM]
3       4                {'NORM': 100.0, 'SR': 0.0}             [NORM]
4       5                {'NORM': 100.0, 'SR': 0.0}             [NORM]
5       6                {'NORM': 100.0, 'SR': 0.0}             [NORM]
6       7                {'NORM': 100.0, 'SR': 0.0}             [NORM]
7       8    {'IMI': 35.0, 'ABQRS': 0.0, 'SR': 0.0}               [MI]
8       9                {'NORM': 100.0, 'SR': 0.0}             [NORM]
9      10                {'NORM': 100.0, 'SR': 0.0}             [NORM]


In [16]:
print("Total ECG records:", len(df))

print("\nDiagnostic class counts:")
print(
    df["diagnostic_classes"]
    .explode()
    .value_counts()
)

print("\nECGs without diagnostic classes:")
print(
    df["diagnostic_classes"]
    .apply(len)
    .eq(0)
    .sum()
)

Total ECG records: 21799

Diagnostic class counts:
diagnostic_classes
NORM    9514
MI      5469
STTC    5235
CD      4898
HYP     2649
Name: count, dtype: int64

ECGs without diagnostic classes:
411


In [17]:
# Select ECGs without diagnostic classes
unmapped_ecgs = df[
    df["diagnostic_classes"].apply(len) == 0
]

print("ECGs without diagnostic classes:", len(unmapped_ecgs))

# Show their original diagnostic codes
print(
    unmapped_ecgs[
        ["ecg_id", "scp_codes"]
    ].head(10)
)

ECGs without diagnostic classes: 411
     ecg_id                                   scp_codes
16       17  {'AFLT': 100.0, 'ABQRS': 0.0, 'AFIB': 0.0}
17       18                             {'AFLT': 100.0}
19       20               {'AFLT': 100.0, 'ABQRS': 0.0}
22       23                             {'AFLT': 100.0}
33       34                             {'AFLT': 100.0}
137     144                             {'PACE': 100.0}
182     190                             {'PACE': 100.0}
374     382                             {'PACE': 100.0}
377     385                             {'PACE': 100.0}
441     449                             {'AFLT': 100.0}


In [18]:
# Inspect the codes in the first 10 unmapped ECGs
for codes in unmapped_ecgs["scp_codes"].head(10):
    for code in codes:
        if code not in diagnostic_scp.index:
            if code in scp.index:
                print(
                    code,
                    "| diagnostic:",
                    scp.loc[code, "diagnostic"],
                    "| class:",
                    scp.loc[code, "diagnostic_class"]
                )
            else:
                print(code, "| not found in scp table")

AFLT | diagnostic: nan | class: nan
ABQRS | diagnostic: nan | class: nan
AFIB | diagnostic: nan | class: nan
AFLT | diagnostic: nan | class: nan
AFLT | diagnostic: nan | class: nan
ABQRS | diagnostic: nan | class: nan
AFLT | diagnostic: nan | class: nan
AFLT | diagnostic: nan | class: nan
PACE | diagnostic: nan | class: nan
PACE | diagnostic: nan | class: nan
PACE | diagnostic: nan | class: nan
PACE | diagnostic: nan | class: nan
AFLT | diagnostic: nan | class: nan


In [19]:
patient_counts = (
    df.groupby("strat_fold")["patient_id"]
    .nunique()
    .sort_index()
)

print("Unique patients in each fold:")
print(patient_counts)

Unique patients in each fold:
strat_fold
1     1878
2     1849
3     1887
4     1883
5     1890
6     1884
7     1871
8     1881
9     1942
10    1904
Name: patient_id, dtype: int64


In [20]:
train_df = df[df["strat_fold"].between(1, 8)].copy()
val_df = df[df["strat_fold"] == 9].copy()
test_df = df[df["strat_fold"] == 10].copy()

print("Training ECGs:", len(train_df))
print("Validation ECGs:", len(val_df))
print("Testing ECGs:", len(test_df))

print("\nUnique patients:")
print("Training:", train_df["patient_id"].nunique())
print("Validation:", val_df["patient_id"].nunique())
print("Testing:", test_df["patient_id"].nunique())

Training ECGs: 17418
Validation ECGs: 2183
Testing ECGs: 2198

Unique patients:
Training: 15023
Validation: 1942
Testing: 1904


In [21]:
train_patients = set(train_df["patient_id"].dropna())
val_patients = set(val_df["patient_id"].dropna())
test_patients = set(test_df["patient_id"].dropna())

print("Training and validation overlap:",
      len(train_patients & val_patients))

print("Training and testing overlap:",
      len(train_patients & test_patients))

print("Validation and testing overlap:",
      len(val_patients & test_patients))

Training and validation overlap: 0
Training and testing overlap: 0
Validation and testing overlap: 0


In [22]:
def count_diagnostic_classes(data):
    return (
        data["diagnostic_classes"]
        .explode()
        .value_counts()
        .reindex(["NORM", "MI", "STTC", "CD", "HYP"], fill_value=0)
    )

print("TRAINING CLASS COUNTS")
print(count_diagnostic_classes(train_df))

print("\nVALIDATION CLASS COUNTS")
print(count_diagnostic_classes(val_df))

print("\nTESTING CLASS COUNTS")
print(count_diagnostic_classes(test_df))

TRAINING CLASS COUNTS
diagnostic_classes
NORM    7596
MI      4379
STTC    4186
CD      3907
HYP     2119
Name: count, dtype: int64

VALIDATION CLASS COUNTS
diagnostic_classes
NORM    955
MI      540
STTC    528
CD      495
HYP     268
Name: count, dtype: int64

TESTING CLASS COUNTS
diagnostic_classes
NORM    963
MI      550
STTC    521
CD      496
HYP     262
Name: count, dtype: int64
